# Run Pipeline On Kaggle (Adapter + sales_data.csv)

This notebook loads a LoRA adapter, generates model output for a query, parses JSON actions, and executes them on sales_data.csv to produce final answer.

In [10]:
# Kaggle compatibility fix for PEFT + torchao
%pip -q uninstall -y torchao

# Runtime dependencies
%pip -q install -U transformers peft accelerate bitsandbytes sentencepiece

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [11]:
import ast
import json
import re
from pathlib import Path

import pandas as pd
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

In [12]:
SYSTEM_PROMPT = (
    "You are a data-analysis tool-planning agent. ",
    "Return ONLY valid JSON with keys: actions, answer. ",
    "actions must be a list of tool calls as strings, and answer must be null."
)

def extract_first_json_object(text):
    decoder = json.JSONDecoder()
    i = 0
    while i < len(text):
        if text[i] != '{':
            i += 1
            continue
        try:
            obj, _ = decoder.raw_decode(text[i:])
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass
        i += 1
    raise ValueError('No valid JSON object found in model output.')

def _get_base_model_name(adapter_dir):
    cfg_path = Path(adapter_dir) / 'adapter_config.json'
    if not cfg_path.exists():
        raise FileNotFoundError(f'adapter_config.json not found in {adapter_dir}')
    with cfg_path.open('r', encoding='utf-8') as f:
        cfg = json.load(f)
    base_name = cfg.get('base_model_name_or_path')
    if not base_name:
        raise ValueError("adapter_config.json missing 'base_model_name_or_path'")
    return base_name

def load_planner_model(adapter_dir):
    adapter_dir = Path(adapter_dir)
    base_model_name = _get_base_model_name(adapter_dir)

    tokenizer = None
    for source in (base_model_name, str(adapter_dir)):
        for use_fast in (True, False):
            try:
                tokenizer = AutoTokenizer.from_pretrained(source, use_fast=use_fast, trust_remote_code=True)
                break
            except Exception:
                continue
        if tokenizer is not None:
            break

    if tokenizer is None:
        raise RuntimeError('Tokenizer could not be loaded from base model or adapter directory.')

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if torch.cuda.is_available():
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            torch_dtype=torch.float16,
            device_map='auto',
            trust_remote_code=True,
        )
        model = PeftModel.from_pretrained(base_model, str(adapter_dir), is_trainable=False)
        device = torch.device('cuda:0')
    else:
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            torch_dtype=torch.float32,
            trust_remote_code=True,
            low_cpu_mem_usage=True,
        )
        model = PeftModel.from_pretrained(base_model, str(adapter_dir), is_trainable=False)
        device = torch.device('cpu')

    model.eval()
    if hasattr(model, 'gradient_checkpointing_disable'):
        model.gradient_checkpointing_disable()
    model.config.use_cache = True

    return model, tokenizer, device

def generate_model_output(model, tokenizer, device, query, max_new_tokens=220):
    messages = [
        {'role': 'system', 'content': ' '.join(SYSTEM_PROMPT)},
        {'role': 'user', 'content': f'Query: {query}\nOutput JSON:'},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=2048).to(device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_only = out_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen_only, skip_special_tokens=True).strip()

def parse_model_output(model_text):
    obj = extract_first_json_object(model_text)
    actions = obj.get('actions', [])
    if not isinstance(actions, list) or not all(isinstance(a, str) for a in actions):
        raise ValueError("Parsed JSON must contain 'actions' as list[str].")
    return obj

def parse_agent_action(action_str):
    action_str = action_str.strip()
    if action_str.startswith('filter_data'):
        m = re.search(r"column='([^']+)'\s*,\s*value=(.+)\)$", action_str)
        if m:
            raw_value = m.group(2)
            try:
                value = ast.literal_eval(raw_value)
            except Exception:
                value = raw_value.strip("'")
            return {'tool': 'filter', 'args': {'column': m.group(1), 'op': '==', 'value': value}}

    if action_str.startswith('group_by'):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {'tool': 'groupby', 'args': {'column': m.group(1)}}

    if action_str.startswith('aggregate_sum'):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {'tool': 'aggregate', 'args': {'column': m.group(1), 'agg': 'sum'}}

    if action_str.startswith('aggregate_mean'):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {'tool': 'aggregate', 'args': {'column': m.group(1), 'agg': 'mean'}}

    if action_str.startswith('aggregate_count'):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {'tool': 'aggregate', 'args': {'column': m.group(1), 'agg': 'count'}}

    if action_str.startswith('sort_by'):
        m = re.search(r"column='([^']+)', order='([^']+)'", action_str)
        if m:
            return {'tool': 'sort', 'args': {'column': m.group(1), 'ascending': m.group(2) != 'desc'}}

    if action_str.startswith('top_k'):
        m = re.search(r"k=(\d+)", action_str)
        if m:
            return {'tool': 'topk', 'args': {'k': int(m.group(1))}}

    raise ValueError(f'Could not parse action: {action_str}')

class ToolExecutor:
    def __init__(self, df):
        self.df = df.copy()

    def execute(self, actions):
        df = self.df
        for step_id, action in enumerate(actions):
            tool = action['tool']
            args = action.get('args', {})
            try:
                if tool == 'filter':
                    df = self._filter(df, **args)
                elif tool == 'groupby':
                    df = self._groupby(df, **args)
                elif tool == 'aggregate':
                    df = self._aggregate(df, **args)
                elif tool == 'sort':
                    df = self._sort(df, **args)
                elif tool == 'topk':
                    df = self._topk(df, **args)
                else:
                    raise ValueError(f'Unknown tool: {tool}')
            except Exception as e:
                raise RuntimeError(f'Error at step {step_id} ({tool}): {str(e)}')
        return df

    def _filter(self, df, column, op, value):
        if op == '==':
            return df[df[column] == value]
        elif op == '>':
            return df[df[column] > value]
        elif op == '<':
            return df[df[column] < value]
        raise ValueError(f'Unsupported op: {op}')

    def _groupby(self, df, column):
        return df.groupby(column)

    def _aggregate(self, df, column, agg):
        if agg == 'sum':
            result = df[column].sum()
        elif agg == 'mean':
            result = df[column].mean()
        elif agg == 'count':
            result = df[column].count()
        else:
            raise ValueError(f'Unsupported aggregation: {agg}')

        if hasattr(result, 'reset_index'):
            return result.reset_index()
        return pd.DataFrame({column: [result]})

    def _sort(self, df, column, ascending=False):
        return df.sort_values(by=column, ascending=ascending)

    def _topk(self, df, k):
        return df.head(k)

def execute_actions_from_list(df, raw_actions):
    parsed_actions = [parse_agent_action(a) for a in raw_actions]
    valid_cols = set(df.columns.astype(str).tolist())
    invalid_cols = [step.get('args', {}).get('column') for step in parsed_actions if step.get('args', {}).get('column') not in valid_cols and step.get('args', {}).get('column') is not None]
    if invalid_cols:
        raise ValueError(f'Unsupported columns in actions: {sorted(set(invalid_cols))}. Available columns: {sorted(valid_cols)}')

    result_df = ToolExecutor(df).execute(parsed_actions)
    if isinstance(result_df, pd.DataFrame):
        if len(result_df) == 1 and len(result_df.columns) == 1:
            return result_df.iloc[0, 0]
        return result_df.to_dict(orient='records')
    return result_df

def run_query_with_model(adapter_dir, csv_path, query, max_new_tokens=220):
    model, tokenizer, device = load_planner_model(adapter_dir)
    raw_output = generate_model_output(model, tokenizer, device, query, max_new_tokens=max_new_tokens)
    parsed_output = parse_model_output(raw_output)
    actions = parsed_output.get('actions', [])
    answer = execute_actions_from_list(pd.read_csv(csv_path), actions)
    final_output = {'actions': actions, 'answer': answer}
    return {'raw_output': raw_output, 'parsed_output': parsed_output, 'final_output': final_output}

In [13]:
# Configure these paths in Kaggle
# Example: '/kaggle/input/<dataset-name>/qwen2.5-1.5b-colab-20260414-092403/adapter'
ADAPTER_DIR = Path('/kaggle/input/datasets/thandaprani/adapter')
CSV_PATH = Path('/kaggle/input/datasets/thandaprani/salesdata/sales_data.csv')
QUERY = 'Top 3 cities by revenue in 2022'
MAX_NEW_TOKENS = 220

In [14]:
result = run_query_with_model(
    adapter_dir=ADAPTER_DIR,
    csv_path=CSV_PATH,
    query=QUERY,
    max_new_tokens=MAX_NEW_TOKENS,
)

print('Raw model output:')
print(result['raw_output'])

print('\nParsed output JSON:')
print(json.dumps(result['parsed_output'], ensure_ascii=False, indent=2))

print('\nFinal output (with computed answer):')
print(json.dumps(result['final_output'], ensure_ascii=False, indent=2))

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Raw model output:
{"actions": ["filter_data(column='year', value=2022)", "group_by(column='city')", "aggregate_sum(column='revenue')", "sort_by(column='revenue', order='desc')", "top_k(k=3)"], "answer": null}

Parsed output JSON:
{
  "actions": [
    "filter_data(column='year', value=2022)",
    "group_by(column='city')",
    "aggregate_sum(column='revenue')",
    "sort_by(column='revenue', order='desc')",
    "top_k(k=3)"
  ],
  "answer": null
}

Final output (with computed answer):
{
  "actions": [
    "filter_data(column='year', value=2022)",
    "group_by(column='city')",
    "aggregate_sum(column='revenue')",
    "sort_by(column='revenue', order='desc')",
    "top_k(k=3)"
  ],
  "answer": [
    {
      "city": "Delhi",
      "revenue": 10735835.01
    },
    {
      "city": "Mumbai",
      "revenue": 10403972.39
    },
    {
      "city": "Bangalore",
      "revenue": 8498093.15
    }
  ]
}
